# Day 9 — Spatial Feature Extraction with CNNs

**Agri-AI Bootcamp** · Phase 3 of 5: *CNNs & Crop Disease* · Days 9–12

> Build a shallow CNN (Conv2d, MaxPool2d, Linear) on healthy vs diseased leaf images.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

A convolution slides a small filter over the image and produces a feature map that lights up where a pattern (edge, texture) appears. Pooling shrinks the map; a final Linear layer classifies the flattened features. Weight sharing keeps CNNs far smaller than fully connected image models.

### 🌾 Agriculture example

Early filters respond to leaf veins and edges; deeper ones respond to yellow halos or brown lesions that separate diseased from healthy leaves.

> **Watch out:** Resize and normalise every image the same way for train and test (128×128 here).

## 2. Setup

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)
print("device:", dev)

## 3. Data

In [ ]:
# ---- Leaf image helper -------------------------------------------------------
# Real course dataset: PlantVillage / Tomato leaf images on Kaggle (needs a
# kaggle.json API token).  So that this notebook runs anywhere, we fall back to
# a small SYNTHETIC leaf set: green patches = "healthy", green patches with
# brown lesions = "diseased".  Swap in the real folder when you have it.
import numpy as np, torch

def synth_leaves(n=400, size=64, seed=0):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, 3, size, size), dtype=np.float32)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        leaf = np.stack([rng.uniform(.10, .25, (size, size)),
                         rng.uniform(.45, .70, (size, size)),
                         rng.uniform(.10, .25, (size, size))])
        diseased = i % 2
        if diseased:                      # add 3-6 brown lesions
            for _ in range(rng.integers(3, 7)):
                cx, cy = rng.integers(8, size - 8, 2)
                r = rng.integers(4, 9)
                yy, xx = np.ogrid[:size, :size]
                m = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
                leaf[0][m], leaf[1][m], leaf[2][m] = .45, .28, .10
        X[i] = leaf
        y[i] = diseased
    return torch.tensor(X), torch.tensor(y)

CLASSES = ["healthy", "diseased"]
X_img, y_img = synth_leaves()
print("images:", tuple(X_img.shape), "| labels:", torch.bincount(y_img).tolist())

# train / test split
perm = torch.randperm(len(y_img))
X_img, y_img = X_img[perm], y_img[perm]
cut = int(.8*len(y_img))
tr_dl = DataLoader(TensorDataset(X_img[:cut], y_img[:cut]), batch_size=32, shuffle=True)
te_dl = DataLoader(TensorDataset(X_img[cut:], y_img[cut:]), batch_size=64)

fig, ax = plt.subplots(1, 6, figsize=(12, 2.2))
for i in range(6):
    ax[i].imshow(X_img[i].permute(1,2,0).numpy()); ax[i].axis("off")
    ax[i].set_title(CLASSES[y_img[i]], fontsize=9)
plt.show()

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
net=nn.Sequential(nn.Conv2d(3,16,3),nn.ReLU(),nn.MaxPool2d(2),
  nn.Conv2d(16,32,3),nn.ReLU(),nn.MaxPool2d(2),
  nn.Flatten(),nn.LazyLinear(2))   # healthy vs diseased
```

## 5. Worked solution

In [ ]:
# ---- A shallow CNN: Conv -> ReLU -> Pool, twice, then a Linear head ---------
class LeafCNN(nn.Module):
    def __init__(self, n_cls=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64 -> 32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))  # 16 -> 8
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64*8*8, 64), nn.ReLU(),
                                  nn.Linear(64, n_cls))
    def forward(self, x): return self.head(self.features(x))

cnn = LeafCNN().to(dev)
print(cnn)
print("\nshape walk-through:")
z = X_img[:1].to(dev)
for layer in cnn.features:
    z = layer(z); print(f"  {layer.__class__.__name__:<10} -> {tuple(z.shape)}")

# parameter count vs a fully connected model on the same image
fc_params = 64*64*3 * 64
print(f"\nCNN params {sum(p.numel() for p in cnn.parameters()):,} | "
      f"one dense layer on raw pixels would already need {fc_params:,} -- weight sharing wins")

# ---- train ------------------------------------------------------------------
opt, lossf = torch.optim.Adam(cnn.parameters(), 1e-3), nn.CrossEntropyLoss()
for epoch in range(8):
    cnn.train()
    for xb, yb in tr_dl:
        opt.zero_grad(); lossf(cnn(xb.to(dev)), yb.to(dev)).backward(); opt.step()
    cnn.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in te_dl:
            c += (cnn(xb.to(dev)).argmax(1).cpu() == yb).sum().item(); t += len(yb)
    print(f"epoch {epoch} test acc {c/t:.3f}")

# ---- What did the first layer learn to look for? ----------------------------
with torch.no_grad():
    fmaps = cnn.features[0](X_img[1:2].to(dev))[0].cpu()
fig, ax = plt.subplots(1, 7, figsize=(13, 2))
ax[0].imshow(X_img[1].permute(1,2,0)); ax[0].set_title("input", fontsize=8); ax[0].axis("off")
for i in range(6):
    ax[i+1].imshow(fmaps[i], cmap="viridis"); ax[i+1].axis("off")
    ax[i+1].set_title(f"filter {i}", fontsize=8)
plt.suptitle("feature maps light up where a pattern appears", fontsize=10); plt.show()

## 6. Your turn

1. Resize the images to 128x128 and fix the head. Which line had to change?
2. Remove the MaxPool layers — count the parameters and watch memory use.
3. Swap in real PlantVillage images: upload kaggle.json, download the dataset, use ImageFolder + the same model.

In [ ]:
# TODO 1: 128x128 -- nn.LazyLinear avoids hand-computing the flattened size
cnn2 = nn.Sequential(nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                     nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                     nn.Flatten(), nn.LazyLinear(2)).to(dev)
big = torch.nn.functional.interpolate(X_img[:4], size=128)
print("LazyLinear output:", cnn2(big.to(dev)).shape)

# TODO 2: no pooling -- compare parameter counts
# TODO 3: real data
# from google.colab import files; files.upload()          # kaggle.json
# !pip -q install kaggle && mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/
# !kaggle datasets download -d emmarex/plantdisease -q && unzip -q plantdisease.zip
# from torchvision.datasets import ImageFolder

## 7. Dataset and references

**Dataset:** [Kaggle: PlantVillage leaf images](https://www.kaggle.com/datasets/emmarex/plantdisease)

**References**

- [Brandon Rohrer: How CNNs work](https://www.youtube.com/watch?v=FmpDIaiMIeA)

**Course material:** Brandon Rohrer: How CNNs work

---
**Next:** Day 10 — Deep Vision Architectures & ResNet